In [ ]:
!nvidia-smi || echo "CPU runtime, which is correct for Day 3"
!python -c "import cv2, sklearn, numpy, matplotlib; print('cv2', cv2.__version__); print('sklearn', sklearn.__version__); print('numpy', numpy.__version__)"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
cd /content
rm -rf pcb-defect
git clone --depth 1 https://github.com/akshatyuvan/pcb-defect.git
cd pcb-defect
git log --oneline -1

In [ ]:
%%bash
cd /content/pcb-defect
rm -rf data artifacts
ln -s /content/drive/MyDrive/pcb-defect/data      data
ln -s /content/drive/MyDrive/pcb-defect/artifacts artifacts
ls -la | grep -E ' data| artifacts'

In [ ]:
%%bash
cd /content/pcb-defect
ls -la data/
ls -la data/patches/
ls -la artifacts/ | head -20
du -sh data/raw/PCBData

In [ ]:
import numpy as np, os
os.chdir('/content/pcb-defect')

for split in ('train', 'val', 'test'):
    z = np.load(f'data/patches/{split}.npz', allow_pickle=True)
    print(f'--- {split} ---')
    print('  keys:', sorted(z.files))
    print('  X', z['X'].shape, z['X'].dtype, '| y', z['y'].shape, z['y'].dtype)
    print('  boards', len(z['board_ids']), '| board_labels', z['board_labels'].shape)
    print('  boxes', len(z['box_class']), '| primary', int(z['box_primary'].sum()))
    print('  sample board_id:', repr(str(z['board_ids'][0])))

In [ ]:
import sys; sys.path.insert(0, '/content/pcb-defect')
from src.data.paths import index_boards, board_key, resolve
import numpy as np

idx = index_boards('data/raw/PCBData')
print('boards found on disk:', len(idx))

z = np.load('data/patches/test.npz', allow_pickle=True)
print('first npz id ->', repr(str(z['board_ids'][0])), '-> key', board_key(z['board_ids'][0]))
pairs = resolve(z['board_ids'], idx)
print('resolved', len(pairs), 'pairs')
print(pairs[0][0]); print(pairs[0][1])

In [ ]:
import cv2, matplotlib.pyplot as plt
from src.baseline.template_diff import signed_diff, clean_mask, copper_mask

t = cv2.imread(str(pairs[0][0]), cv2.IMREAD_GRAYSCALE)
r = cv2.imread(str(pairs[0][1]), cv2.IMREAD_GRAYSCALE)
print('copper fraction, test:', copper_mask(t).mean(), '| template:', copper_mask(r).mean())

for radius in (0, 4):
    extra, missing, shift = signed_diff(t, r, radius)
    print(f'radius {radius}: shift {shift}, extra {extra.sum()}, missing {missing.sum()}')

extra, missing, shift = signed_diff(t, r, 4)
fig, ax = plt.subplots(1, 5, figsize=(20, 4))
for a, im, ti in zip(ax, [t, r, extra*255, missing*255,
                          clean_mask(extra, 5, 10)*255 + clean_mask(missing, 5, 10)*255],
                     ['test', 'template', 'extra copper (raw)', 'missing copper (raw)',
                      'both, cleaned k=5 area=10']):
    a.imshow(im, cmap='gray', vmin=0, vmax=255); a.set_title(ti); a.axis('off')
plt.show()

In [ ]:
%%bash
cd /content/pcb-defect
python -m src.baseline.template_diff \
  --patches data/patches \
  --raw data/raw/PCBData \
  --artifacts artifacts \
  --target-recall 0.97 \
  --align-radius 4 \
  --cnn-ckpt artifacts/r4_weighted_p05_registered_best.pt \
  --device cpu \
  --tag day3_baseline 2>&1 | tail -80

In [ ]:
print(open('artifacts/day3_baseline_report.txt').read())

In [ ]:
from IPython.display import Image, display
display(Image('artifacts/figures/day3_baseline_pr.png'))
display(Image('artifacts/figures/day3_baseline_examples.png'))

In [ ]:
%%bash
cd /content/pcb-defect/artifacts
zip -r /content/drive/MyDrive/pcb-defect/day3_artifacts.zip \
  day3_baseline_report.txt day3_baseline_config.json \
  figures/day3_baseline_pr.png figures/day3_baseline_examples.png
ls -lh /content/drive/MyDrive/pcb-defect/day3_artifacts.zip

In [ ]:
%%bash
ls -la /content/drive/MyDrive/pcb-defect/artifacts/ | grep day3
ls -la /content/drive/MyDrive/pcb-defect/day3_artifacts.zip 2>&1